In [ ]:
import pandas as pd
from tabulate import tabulate

# SFT (w/o reasoning traces): fine-tune directly on ViNumQA train.json, target
# is the bare program string (no chain-of-thought), unlike HUSTUET's
# knowledge-distillation approach in the paper. Uses the official 3-way split.
train_df = pd.read_json('/kaggle/input/datasets/ntphuc149x2/vlsp2025-vinumqa/train.json')
valid_df = pd.read_json('/kaggle/input/datasets/ntphuc149x2/vlsp2025-vinumqa/valid.json')
test_df = pd.read_json('/kaggle/input/datasets/ntphuc149x2/vlsp2025-vinumqa/test.json')
print(f"train={len(train_df)}, valid={len(valid_df)}, test={len(test_df)}")

In [ ]:
def formatting_pre_text(sample):
    return "\n".join(sample["pre_text"])

def formatting_table(sample):
    return tabulate(sample["table"][1:], headers=sample["table"][0], tablefmt="github")

def formatting_post_text(sample):
    return "\n".join(sample["post_text"])

def processing_input_question(sample):
    return sample["qa"]["question"]

def processing_program_content(sample):
    return sample["qa"]["program"]

def processing_answer_content(sample):
    return sample["qa"]["exe_ans"]

def process_split(df):
    df = df.copy()
    df["pre_text_processed"] = df.apply(formatting_pre_text, axis=1)
    df["post_text_processed"] = df.apply(formatting_post_text, axis=1)
    df["table_processed"] = df.apply(formatting_table, axis=1)
    df["input_question"] = df.apply(processing_input_question, axis=1)
    df["program_processed"] = df.apply(processing_program_content, axis=1)
    df["answer_processed"] = df.apply(processing_answer_content, axis=1)
    df = df[["pre_text_processed", "table_processed", "post_text_processed", "input_question", "program_processed", "answer_processed"]]
    df.columns = ["pre_text", "table", "post_text", "question", "program", "answer"]
    return df

train_df = process_split(train_df)
valid_df = process_split(valid_df)
test_df = process_split(test_df)
test_df["generated_program"] = ""

train_df.sample(n=3)

In [ ]:
SYSTEM_MESSAGE = """You are a financial analysis AI. Your task is to generate a sequential computation program to answer the question, based on the provided context.
 
### LIST OF 10 VALID OPERATORS:
 
1. add(a, b) -> a + b
2. subtract(a, b) -> a - b
3. multiply(a, b) -> a * b
4. divide(a, b) -> a / b
5. exp(a, b) -> a^b
6. greater(a, b) -> 1.0 if a > b, else 0.0
7. table_sum(val1, val2, val3, ...) -> sum of the values
8. table_average(val1, val2, val3, ...) -> arithmetic mean of the values
9. table_max(val1, val2, val3, ...) -> maximum of the values
10. table_min(val1, val2, val3, ...) -> minimum of the values
 
### RULES:
- Do not use free-form mathematical symbols ("+", "-", "*", "/") outside of parentheses. Every calculation must use one of the 10 operators above.
- Do not use square brackets `[]` inside table-type functions (e.g. write table_max(1, 2, 3), not table_max([1, 2, 3])).
- Do not perform mental calculations or provide explanations. The output must contain only the program string.
- Reference the result of a previous step using #0 (step 1), #1 (step 2), etc. Steps are separated by commas.
- Preserve the original number format from the context. If a value is missing, use 'none'."""

USER_MESSAGE_FRAME = """### CONTEXT:
[TEXT BEFORE TABLE]
{pre_text}
 
[TABLE]
{table}
 
[TEXT AFTER TABLE]
{post_text}
 
### QUESTION:
{question}
 
### PROGRAM:"""

# Same prompt format as the 0-shot/1-shot notebooks, so results are
# comparable across the three experiments (only the training regime differs).

In [ ]:
!pip install -q -U peft trl

In [ ]:
import os
import torch
from huggingface_hub import login

# Read the HF token from a Kaggle Secret / env var instead of hardcoding it.
# On Kaggle: Add-ons > Secrets > add "HF_TOKEN", then attach it to this notebook.
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN not found. Add it as a Kaggle Secret (Add-ons > Secrets) "
        "or set the HF_TOKEN environment variable before running this notebook."
    )

login(HF_TOKEN)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",
)
base_model.config.use_cache = False  # required for gradient checkpointing during training

# LoRA (PEFT): train only small adapter matrices instead of all 4B params --
# fits comfortably on a single free-tier Kaggle GPU (T4/P100), unlike full
# fine-tuning which would need far more VRAM for optimizer states.
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

In [ ]:
from datasets import Dataset

MAX_SEQ_LEN = 2048

def build_prompt(row):
    user_msg = USER_MESSAGE_FRAME.format(
        pre_text=row["pre_text"], table=row["table"],
        post_text=row["post_text"], question=row["question"],
    )
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": user_msg},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )

def tokenize_example(row):
    prompt = build_prompt(row)
    # Target is the bare program string only -- no reasoning trace -- per the
    # "w/o reasoning traces" spec for this SFT baseline.
    completion = str(row["program"]).strip() + tokenizer.eos_token

    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    completion_ids = tokenizer(completion, add_special_tokens=False)["input_ids"]

    input_ids = (prompt_ids + completion_ids)[:MAX_SEQ_LEN]
    # Mask the prompt portion so loss is only computed on the completion
    # (the program string) -- standard SFT practice, the model isn't
    # penalized for "predicting" the context/question it was given.
    labels = ([-100] * len(prompt_ids) + completion_ids)[:MAX_SEQ_LEN]
    attention_mask = [1] * len(input_ids)

    pad_len = MAX_SEQ_LEN - len(input_ids)
    if pad_len > 0:
        input_ids = input_ids + [tokenizer.pad_token_id] * pad_len
        labels = labels + [-100] * pad_len
        attention_mask = attention_mask + [0] * pad_len

    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

train_records = [tokenize_example(row) for _, row in train_df.iterrows()]
valid_records = [tokenize_example(row) for _, row in valid_df.iterrows()]

train_dataset = Dataset.from_list(train_records)
valid_dataset = Dataset.from_list(valid_records)
print(f"train_dataset={len(train_dataset)}, valid_dataset={len(valid_dataset)}")

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

OUTPUT_DIR = "/kaggle/working/qwen3-4b-vinumqa-sft"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,   # effective batch size = 16
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    bf16=True,
    gradient_checkpointing=True,
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=False),
)

trainer.train()

In [ ]:
ADAPTER_DIR = "/kaggle/working/qwen3-4b-vinumqa-sft-adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Saved LoRA adapter to {ADAPTER_DIR}")

In [ ]:
import gc
from pathlib import Path
from tqdm import tqdm

model.eval()
model.config.use_cache = True  # re-enable KV cache for faster generation (was off for training)

# Checkpointed inference loop on the held-out test set -- same pattern as the
# 0-shot/1-shot notebooks, so a Kaggle session interruption doesn't lose
# already-generated predictions.
CHECKPOINT_PATH = Path("outputs/sft_qwen3-4b_checkpoint.csv")
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
CHECKPOINT_EVERY = 20

if CHECKPOINT_PATH.exists():
    ckpt = pd.read_csv(CHECKPOINT_PATH).fillna("")
    test_df.loc[ckpt.index, "generated_program"] = ckpt["generated_program"].values
    print(f"Resumed {(test_df['generated_program'] != '').sum()} / {len(test_df)} samples from checkpoint.")

todo_idx = test_df.index[test_df["generated_program"] == ""]
for i, df_index in enumerate(tqdm(todo_idx, desc="Generating program...")):
    row = test_df.loc[df_index]
    prompt = build_prompt(row)
    model_inputs = tokenizer([prompt], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(**model_inputs, max_new_tokens=512)
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

    try:
        index = len(output_ids) - output_ids[::-1].index(151668)  # Qwen3 </think> token id
    except ValueError:
        index = 0

    output = tokenizer.decode(output_ids[index:], skip_special_tokens=True)
    test_df.at[df_index, "generated_program"] = output.strip()

    gc.collect()
    torch.cuda.empty_cache()

    if (i + 1) % CHECKPOINT_EVERY == 0:
        test_df.to_csv(CHECKPOINT_PATH, index=False)

test_df.to_csv(CHECKPOINT_PATH, index=False)
print(f"Done. {(test_df['generated_program'] != '').sum()} / {len(test_df)} samples generated.")

In [ ]:
"""
Parser + evaluation utilities for FinQA/VLSP-2025 style computation programs.
(Same implementation as the 0-shot/1-shot notebooks, including the fix for
percent literals: "16.63%" -> 0.1663, not just stripped to 16.63.)
"""

import re
from typing import List, Tuple, Union, Optional


VALID_OPERATORS = {
    "add", "subtract", "multiply", "divide", "exp", "greater",
    "table_sum", "table_average", "table_max", "table_min",
}

_BINARY_OPS = {"add", "subtract", "multiply", "divide", "exp", "greater"}
_TABLE_OPS = {"table_sum", "table_average", "table_max", "table_min"}

_NUM_RE = re.compile(r"^-?\d+(\.\d+)?$")
_REF_RE = re.compile(r"^#(\d+)$")


def _parse_numeric_literal(raw: str) -> float:
    cleaned = raw.replace(",", "").replace("$", "").strip()
    is_percent = cleaned.endswith("%")
    if is_percent:
        cleaned = cleaned[:-1].strip()
    if not _NUM_RE.match(cleaned):
        raise ValueError(f"Cannot parse numeric argument: '{raw}'")
    value = float(cleaned)
    return value / 100.0 if is_percent else value


def extract_program(raw_text: str) -> str:
    text = raw_text.strip()
    text = re.sub(r"```[a-zA-Z]*", "", text)
    text = text.replace("```", "").strip()

    op_names = "|".join(sorted(VALID_OPERATORS, key=len, reverse=True))
    pattern = rf"\b(?:{op_names})\([^()]*\)"
    matches = re.findall(pattern, text)

    if matches:
        return ", ".join(m.strip() for m in matches)
    return text


def _split_top_level(s: str, sep: str = ",") -> List[str]:
    parts = []
    depth = 0
    current = []
    for ch in s:
        if ch == "(":
            depth += 1
            current.append(ch)
        elif ch == ")":
            depth -= 1
            current.append(ch)
        elif ch == sep and depth == 0:
            parts.append("".join(current))
            current = []
        else:
            current.append(ch)
    if current:
        parts.append("".join(current))
    return [p.strip() for p in parts if p.strip() != ""]


def parse_program(program_str: str) -> List[Tuple[str, List[str]]]:
    program_str = program_str.strip().rstrip(",").strip()
    if not program_str:
        raise ValueError("Empty program string.")

    steps: List[Tuple[str, List[str]]] = []
    call_pattern = re.compile(r"\s*([a-zA-Z_]+)\(([^()]*)\)\s*")
    pos = 0
    text = program_str
    while pos < len(text):
        m = call_pattern.match(text, pos)
        if not m:
            raise ValueError(f"Malformed program near: '{text[pos:pos+30]}...'")
        op = m.group(1).strip()
        if op not in VALID_OPERATORS:
            raise ValueError(f"Unknown operator: '{op}'")
        args = _split_top_level(m.group(2), sep=",")
        steps.append((op, args))
        pos = m.end()
        if pos < len(text) and text[pos] == ",":
            pos += 1

    if not steps:
        raise ValueError("No valid steps parsed.")
    return steps


def _resolve_arg(arg: str, results: List[float]) -> Optional[float]:
    arg = arg.strip()

    ref_match = _REF_RE.match(arg)
    if ref_match:
        idx = int(ref_match.group(1))
        if idx >= len(results):
            raise ValueError(f"Reference #{idx} used before step {idx} was computed.")
        return results[idx]

    if arg.lower() == "none":
        return None

    return _parse_numeric_literal(arg)


def execute_program(program_str: str) -> Tuple[float, List[float]]:
    steps = parse_program(program_str)
    results: List[float] = []

    for op, raw_args in steps:
        vals = [_resolve_arg(a, results) for a in raw_args]

        if op in _BINARY_OPS:
            if len(vals) != 2:
                raise ValueError(f"Operator '{op}' expects 2 args, got {len(vals)}.")
            a, b = vals
            if a is None or b is None:
                raise ValueError(f"Operator '{op}' received a 'none' operand.")
            if op == "add":
                r = a + b
            elif op == "subtract":
                r = a - b
            elif op == "multiply":
                r = a * b
            elif op == "divide":
                if b == 0:
                    raise ValueError("Division by zero.")
                r = a / b
            elif op == "exp":
                r = a ** b
            elif op == "greater":
                r = 1.0 if a > b else 0.0

        elif op in _TABLE_OPS:
            if len(vals) < 1:
                raise ValueError(f"Operator '{op}' requires at least 1 arg.")
            if any(v is None for v in vals):
                raise ValueError(f"Operator '{op}' received a 'none' operand.")
            if op == "table_sum":
                r = sum(vals)
            elif op == "table_average":
                r = sum(vals) / len(vals)
            elif op == "table_max":
                r = max(vals)
            elif op == "table_min":
                r = min(vals)
        else:
            raise ValueError(f"Unknown operator: '{op}'")

        results.append(r)

    return results[-1], results


def _normalize_program(program_str: str) -> List[Tuple[str, Tuple[str, ...]]]:
    steps = parse_program(program_str)
    normalized = []
    for op, args in steps:
        norm_args = []
        for a in args:
            a = a.strip()
            if _REF_RE.match(a) or a.lower() == "none":
                norm_args.append(a.lower())
            else:
                try:
                    norm_args.append(f"{round(_parse_numeric_literal(a), 6)}")
                except ValueError:
                    norm_args.append(a)
        normalized.append((op, tuple(norm_args)))
    return normalized


def compute_program_accuracy(
    generated_program: str,
    gold_program: str,
    alternative_gold_programs: Optional[List[str]] = None,
) -> float:
    try:
        gen_norm = _normalize_program(generated_program)
    except ValueError:
        return 0.0

    gold_candidates = [gold_program] + (alternative_gold_programs or [])
    for gold in gold_candidates:
        try:
            gold_norm = _normalize_program(gold)
        except ValueError:
            continue
        if gen_norm == gold_norm:
            return 1.0
    return 0.0


def compute_execution_accuracy(
    generated_program: str,
    gold_answer: Union[str, float],
    rel_tol: float = 1e-3,
    abs_tol: float = 1e-4,
) -> float:
    try:
        predicted, _ = execute_program(generated_program)
    except (ValueError, ZeroDivisionError, OverflowError):
        return 0.0

    try:
        gold = _parse_numeric_literal(str(gold_answer))
    except ValueError:
        return 0.0

    if abs(predicted - gold) <= max(abs_tol, rel_tol * abs(gold)):
        return 1.0
    return 0.0


def evaluate_dataframe(
    df,
    generated_col: str = "generated_program",
    gold_program_col: str = "program",
    gold_answer_col: str = "answer",
    extract_first: bool = True,
):
    df = df.copy()
    pa_scores = []
    ea_scores = []

    for _, row in df.iterrows():
        raw_generated = row[generated_col]
        generated = extract_program(raw_generated) if extract_first else raw_generated
        gold_program = row[gold_program_col]
        gold_answer = row[gold_answer_col]

        pa = compute_program_accuracy(generated, gold_program)
        ea = compute_execution_accuracy(generated, gold_answer)

        pa_scores.append(pa)
        ea_scores.append(ea)

    df["pa_score"] = pa_scores
    df["ea_score"] = ea_scores

    summary = {
        "program_accuracy": sum(pa_scores) / len(pa_scores) if pa_scores else 0.0,
        "execution_accuracy": sum(ea_scores) / len(ea_scores) if ea_scores else 0.0,
    }
    return df, summary


df_scored, summary = evaluate_dataframe(test_df)
print(summary)

In [ ]:
import json

results_path = Path("outputs/sft_qwen3-4b_results.csv")
summary_path = Path("outputs/sft_qwen3-4b_summary.json")

df_scored.to_csv(results_path, index=False)
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump({
        "model": MODEL_NAME,
        "method": "SFT (LoRA, w/o reasoning traces)",
        "train_size": len(train_dataset),
        "epochs": training_args.num_train_epochs,
        "max_seq_len": MAX_SEQ_LEN,
        **summary,
    }, f, ensure_ascii=False, indent=2)

print(f"Saved per-sample results to {results_path}")
print(f"Saved summary to {summary_path}")